In [1]:
from pathlib import Path
import time
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import sys
sys.path.append(str(Path.cwd().parent))

In [2]:
from src.Geo_OneMapClient import BASE, OneMapClient
from src.Geo_Distances import (
    CITY_LAT,
    CITY_LON,
    haversine_m,
    distance_to_city,
    nearest_mrt
)
from src.Geo_CacheBuild import (
    raw_path,
    processed_path,
    aid_path,
    build_unique_addresses,
    build_zone_name_to_id,
    build_planning_polygons,
    lookup_subzone,
    build_geocode_cache,
    build_mrt_from_exit_geojson,
    attach_mrt_to_cache
)

In [3]:
# transform the interim resale transactions csv file to ready-for-search form and save
data_path = f'{processed_path}\\resale_transactions.csv'
unique = build_unique_addresses(data_path)
unique

Unique addresses: 9740 -> E:\AI study\HDB_resale_market_analysis\src\..\data\aid\unique_hdb_addresses.csv


,blk,st,search_val
0,406,ANG MO KIO AVE 10,406 ANG MO KIO AVE 10
1,108,ANG MO KIO AVE 4,108 ANG MO KIO AVE 4
2,602,ANG MO KIO AVE 5,602 ANG MO KIO AVE 5
3,465,ANG MO KIO AVE 10,465 ANG MO KIO AVE 10
4,601,ANG MO KIO AVE 5,601 ANG MO KIO AVE 5
...,...,...,...
9735,111B,ALKAFF CRES,111B ALKAFF CRES
9736,112B,ALKAFF CRES,112B ALKAFF CRES
9737,110A,BIDADARI PK DR,110A BIDADARI PK DR
9738,107B,BIDADARI PK DR,107B BIDADARI PK DR


In [4]:
# create a class object and make the cache
flag = False
if flag:
    email = 'XU0033NG@e.ntu.edu.sg'
    password = 'N-7S&Eh-fD5k&Rs'
    client = OneMapClient(email = email, password = password, token = None)
    cache = build_geocode_cache(client, year = 2019, sleep_s = 0.5)
    cache

In [5]:
# obtain the mrt dataset
mrt = build_mrt_from_exit_geojson()
mrt

MRT points: 613 -> E:\AI study\HDB_resale_market_analysis\src\..\data\aid\mrt_stations_cleaned.csv
Unique stations: 190


,name,lat,lon
0,SPRINGLEAF MRT STATION,1.398787,103.818003
1,LENTOR MRT STATION,1.385891,103.835823
2,LENTOR MRT STATION,1.385410,103.835360
3,LENTOR MRT STATION,1.384602,103.837398
4,LENTOR MRT STATION,1.383628,103.837339
...,...,...,...
608,TAN KAH KEE MRT STATION,1.325333,103.807799
609,CASHEW MRT STATION,1.369833,103.764234
610,SIXTH AVENUE MRT STATION,1.330780,103.796566
611,KING ALBERT PARK MRT STATION,1.335537,103.783030


In [6]:
# final step of enrichment
cache = attach_mrt_to_cache()
cache

Attached MRT columns -> E:\AI study\HDB_resale_market_analysis\src\..\data\aid\hdb_geocode_cache.csv
            to_mrt       to_city
count  9740.000000   9740.000000
mean    576.545281  12499.778411
std     373.514402   4373.117199
min      14.659649    590.572958
25%     293.924784   9716.920289
50%     501.866697  13438.980721
75%     770.463641  15608.490475
max    3544.504274  20222.271564


,blk,st,search_val,lat,lon,postal,subzone,zone_id,nearest_mrt,mrt_lat,mrt_lon,to_mrt,to_city,status
0,406,ANG MO KIO AVE 10,406 ANG MO KIO AVE 10,1.362005,103.853880,560406,ANG MO KIO,113,ANG MO KIO MRT STATION,1.369465,103.849939,938.105933,8677.742922,ok
1,108,ANG MO KIO AVE 4,108 ANG MO KIO AVE 4,1.370966,103.838202,560108,ANG MO KIO,113,MAYFLOWER MRT STATION,1.371276,103.836796,160.046540,9782.556462,ok
2,602,ANG MO KIO AVE 5,602 ANG MO KIO AVE 5,1.380709,103.835368,560602,ANG MO KIO,113,LENTOR MRT STATION,1.383628,103.837339,391.564837,10902.032421,ok
3,465,ANG MO KIO AVE 10,465 ANG MO KIO AVE 10,1.366201,103.857201,560465,ANG MO KIO,113,ANG MO KIO MRT STATION,1.369465,103.849939,885.083823,9162.282986,ok
4,601,ANG MO KIO AVE 5,601 ANG MO KIO AVE 5,1.381041,103.835132,NIL,ANG MO KIO,113,LENTOR MRT STATION,1.383628,103.837339,378.015727,10942.845222,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9735,111B,ALKAFF CRES,111B ALKAFF CRES,1.335005,103.873331,342111,TOA PAYOH,162,WOODLEIGH MRT STATION,1.338511,103.870941,471.707005,6168.933205,ok
9736,112B,ALKAFF CRES,112B ALKAFF CRES,1.335862,103.873504,342112,TOA PAYOH,162,WOODLEIGH MRT STATION,1.338511,103.870941,409.782290,6264.156770,ok
9737,110A,BIDADARI PK DR,110A BIDADARI PK DR,1.333484,103.873147,341110,TOA PAYOH,162,POTONG PASIR MRT STATION,1.332801,103.868966,470.911331,6005.594696,ok
9738,107B,BIDADARI PK DR,107B BIDADARI PK DR,1.334372,103.872117,342107,TOA PAYOH,162,POTONG PASIR MRT STATION,1.332801,103.868966,391.361056,6051.845685,ok
